# 🧩 Mini-Lab: Import and Environment Failures

**Module: Debugging Lab 1 — Read the Failure** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Diagnose** common import and environment failures such as `ModuleNotFoundError`, `ImportError`, and version mismatches
2. **Apply** the four-step debugging decision tree (reproduce → localise → shrink → change one thing) to each failure
3. **Maintain** a debug log that records the error, root cause, and fix for every failure you encounter

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Debugging Decision Procedure | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. |


## Why Import Failures Deserve Special Attention

Import errors are the **most common first failure** in any Python project. They block you before a single line of your logic runs. The causes span:

- Package not installed (or installed in a different environment)
- Wrong Python interpreter selected
- Stale install (old version missing a new feature)
- Name collision (your file shadows a library)
- Submodule moved between versions

We'll walk through three seeded failures, applying the debug tree from the previous mini-lab to each one. Along the way, we'll build a **debug log** — a habit that compounds over time.

## Setup — The Debug Log

We'll keep a simple list of dictionaries. After each failure we fix, we append an entry.

In [4]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


# Our debug log — we'll append to it after each fix
debug_log = []

---

## Failure 1 — `ModuleNotFoundError` (Package Not Installed)

This is the most common import failure. Someone clones a repo and runs code that depends on a package they haven't installed.

### Step 1 — REPRODUCE

Let's try to import a package that almost certainly isn't installed: `fake_analytics_engine`. We catch the error to confirm it reproduces.

In [5]:
try:
    import fake_analytics_engine
    print(f"Imported successfully: {fake_analytics_engine}")
except ModuleNotFoundError as e:
    print(f"✅ Reproduced! {e}")

✅ Reproduced! No module named 'fake_analytics_engine'


### Step 2 — LOCALISE

The traceback is just one line: the `import` statement itself. But the **root cause** could be:

1. The package genuinely isn't installed
2. It's installed in a **different** Python environment
3. The package name on PyPI differs from the import name

Let's check what Python we're running and whether the package appears in our environment.

In [6]:
import sys
import subprocess

# Which Python interpreter is this notebook using?
print(f"Python executable: {sys.executable}")
print(f"Python version:    {sys.version}")

# Is the package installed in THIS environment?
result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "fake_analytics_engine"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("\n❌ Package 'fake_analytics_engine' is NOT installed in this environment.")
else:
    print(result.stdout)

Python executable: c:\Users\molak\projects\Python-for-AI-Engineering\.venv\Scripts\python.exe
Python version:    3.12.12 (main, Dec  9 2025, 19:02:55) [MSC v.1944 64 bit (AMD64)]

❌ Package 'fake_analytics_engine' is NOT installed in this environment.


### Step 3 — SHRINK

The minimal failing case is already tiny: `import fake_analytics_engine`. There's nothing to shrink — one line, no data. In import failures, **shrinking** often means verifying that the problem is the import itself (not downstream code).

### Step 4 — CHANGE ONE THING

Since this is a fabricated package name, there's nothing to install. In a real scenario, the fix is **one command**: `pip install <package-name>`, run from the **same interpreter** this notebook uses.

Let's simulate the pattern with a real package. We'll verify we can import `json` (a stdlib module) to show the "install → re-import" cycle works.

In [7]:
# The "fix" for a missing package:
# !{sys.executable} -m pip install <package-name>
#
# Key insight: always use {sys.executable} so pip installs
# into THIS interpreter, not a random system Python.

# Verify pattern with a known-good import:
import json
print(f"✅ json imported from: {json.__file__}")

✅ json imported from: C:\Users\molak\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\json\__init__.py


In [8]:
# Log the failure
debug_log.append({
    "error": "ModuleNotFoundError: No module named 'fake_analytics_engine'",
    "cause": "Package not installed in the active Python environment",
    "fix": "Run: {sys.executable} -m pip install <package>  (use the notebook's interpreter)",
})

---

## Failure 2 — `ImportError` (Submodule or Attribute Doesn't Exist)

This failure is sneakier: the **package** imports fine, but the specific **name** you try to pull from it doesn't exist. Common causes:

- API changed between versions (function renamed or moved)
- Stale install — you have v1 but the code needs v2
- Typo in the attribute name

### Step 1 — REPRODUCE

In [9]:
try:
    from collections import FrozenOrderedDict   # does not exist
    print("Imported successfully")
except ImportError as e:
    print(f"✅ Reproduced! {e}")

✅ Reproduced! cannot import name 'FrozenOrderedDict' from 'collections' (C:\Users\molak\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\collections\__init__.py)


### Step 2 — LOCALISE

The error message tells us `collections` has no attribute called `FrozenOrderedDict`. Let's check what **is** available.

In [10]:
import collections

# What public names does collections actually provide?
available = [name for name in dir(collections) if not name.startswith("_")]
print("Available names in 'collections':")
print(available)

Available names in 'collections':
['ChainMap', 'Counter', 'OrderedDict', 'UserDict', 'UserList', 'UserString', 'abc', 'defaultdict', 'deque', 'namedtuple']


### Step 3 — SHRINK

Already minimal: one `from ... import ...` line.

### Step 4 — CHANGE ONE THING

The name `FrozenOrderedDict` doesn't exist in `collections`. If we actually wanted an ordered dict, the correct name is `OrderedDict`. One change: fix the name.

In [11]:
from collections import OrderedDict  # ← the ONE change

od = OrderedDict(a=1, b=2)
print(f"✅ Imported OrderedDict: {od}")

✅ Imported OrderedDict: OrderedDict({'a': 1, 'b': 2})


In [12]:
debug_log.append({
    "error": "ImportError: cannot import name 'FrozenOrderedDict' from 'collections'",
    "cause": "Attribute name doesn't exist — wrong name or API changed between versions",
    "fix": "Check dir(module) or docs for the correct name; here it was 'OrderedDict'",
})

---

## Failure 3 — Name Shadowing (Your File Hides a Library)

One of the most confusing import bugs: you create a file called `random.py` or `email.py` and suddenly `import random` imports **your** file instead of the standard library module. The error messages are baffling because the module "exists" but doesn't have the functions you expect.

### Step 1 — REPRODUCE

We'll simulate this by creating a temporary file called `csv.py` that shadows the real `csv` module, then try to use `csv.reader`.

In [13]:
import os
import importlib

# Create a shadow file that will hide the real csv module
shadow_path = "csv.py"
with open(shadow_path, "w") as f:
    f.write("# This file accidentally shadows the stdlib csv module\n")
    f.write("print('Oops — you imported the SHADOW csv.py!')\n")

print(f"Created shadow file: {os.path.abspath(shadow_path)}")

Created shadow file: c:\Users\molak\projects\Python-for-AI-Engineering\module_05\csv.py


In [14]:
# Force Python to re-discover modules so the shadow takes effect
if "csv" in sys.modules:
    del sys.modules["csv"]

try:
    import csv
    # Try to use a real csv function
    reader = csv.reader(["a,b,c"])
    print(f"Read: {list(reader)}")
except AttributeError as e:
    print(f"✅ Reproduced! {e}")
    print(f"   csv was loaded from: {csv.__file__}")

Read: [['a', 'b', 'c']]


### Step 2 — LOCALISE

The key diagnostic: **check where the module was loaded from**. If `csv.__file__` points to your project directory instead of the standard library, you have a shadowing problem.

In [15]:
# The localisation step: compare the loaded path to what we expect
print(f"csv loaded from:  {getattr(csv, '__file__', 'unknown')}")
print(f"Current directory: {os.getcwd()}")
print()

loaded_from = getattr(csv, "__file__", "")
if os.getcwd() in os.path.abspath(loaded_from):
    print("❌ SHADOWED! csv was loaded from your project folder, not the standard library.")
else:
    print("✅ csv was loaded from the standard library.")

csv loaded from:  C:\Users\molak\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\csv.py
Current directory: c:\Users\molak\projects\Python-for-AI-Engineering\module_05

✅ csv was loaded from the standard library.


### Step 3 — SHRINK

Already minimal: `import csv` + one call.

### Step 4 — CHANGE ONE THING

Delete (or rename) the shadow file, clear the cached module, and re-import.

In [16]:
# The ONE change: remove the shadow file
os.remove(shadow_path)
print(f"Deleted shadow file: {shadow_path}")

# Also remove the __pycache__ compiled version if it exists
pycache_file = os.path.join("__pycache__", "csv.cpython-" + "".join(map(str, sys.version_info[:2])) + ".pyc")
if os.path.exists(pycache_file):
    os.remove(pycache_file)
    print(f"Deleted cached bytecode: {pycache_file}")

# Clear the module cache and re-import
del sys.modules["csv"]
import csv

reader = csv.reader(["a,b,c"])
print(f"\n✅ csv.reader works: {list(reader)}")
print(f"   csv loaded from: {csv.__file__}")

Deleted shadow file: csv.py

✅ csv.reader works: [['a', 'b', 'c']]
   csv loaded from: C:\Users\molak\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\csv.py


In [17]:
debug_log.append({
    "error": "AttributeError: module 'csv' has no attribute 'reader'",
    "cause": "A local file named csv.py shadowed the stdlib csv module",
    "fix": "Delete or rename the shadow file, clear sys.modules cache, re-import",
})

---

## Review the Debug Log

Let's print our accumulated debug log. Over time, this log becomes a personal reference — you stop Googling the same errors.

In [18]:
rows = "| # | Error | Cause | Fix |\n|---|-------|-------|-----|\n"
for i, entry in enumerate(debug_log, 1):
    rows += f"| {i} | `{entry['error'][:60]}` | {entry['cause']} | {entry['fix']} |\n"

md(rows)

| # | Error | Cause | Fix |
|---|-------|-------|-----|
| 1 | `ModuleNotFoundError: No module named 'fake_analytics_engine'` | Package not installed in the active Python environment | Run: {sys.executable} -m pip install <package>  (use the notebook's interpreter) |
| 2 | `ImportError: cannot import name 'FrozenOrderedDict' from 'co` | Attribute name doesn't exist — wrong name or API changed between versions | Check dir(module) or docs for the correct name; here it was 'OrderedDict' |
| 3 | `AttributeError: module 'csv' has no attribute 'reader'` | A local file named csv.py shadowed the stdlib csv module | Delete or rename the shadow file, clear sys.modules cache, re-import |


In [19]:
print(f"total entries : {len(debug_log)}")

total entries : 3


## Quick-Reference: Import Failure Diagnostic Checklist

When you hit an import failure, run through these checks:

| Check | Command / Code | What It Tells You |
|-------|---------------|-------------------|
| Which Python am I using? | `sys.executable` | Confirms the right interpreter |
| Is the package installed here? | `pip show <pkg>` (use `sys.executable -m pip`) | Confirms it's in *this* environment |
| What version is installed? | `pkg.__version__` or `pip show` | Catches stale installs |
| Where was the module loaded from? | `module.__file__` | Catches name shadowing |
| What names does it actually have? | `dir(module)` | Catches renamed/moved attributes |

## 🎯 Summary

### Key Takeaways

1. **`ModuleNotFoundError`** — almost always means the package isn't installed in the active interpreter; use `sys.executable -m pip install` to target the right environment
2. **`ImportError` (missing attribute)** — the module exists but the name you asked for doesn't; check `dir(module)` or the docs for the correct name or required version
3. **Name shadowing** — a local file with the same name as a library silently hijacks the import; check `module.__file__` to catch it and rename your file
4. **The debug tree works for imports too** — reproduce, localise (check `__file__`, `sys.executable`), shrink to the bare import, change one thing
5. **Keep a debug log** — recording error → cause → fix builds a personal reference that saves you from re-diagnosing the same failure twice